# Whisper on the 219 long clips (GPU)

Kernel: **NeMo ASR** (`~/nemoenv`). Run it in this folder (`recognizers/`) after `NB_v10_asr.ipynb`.

In `NB_v10_asr.ipynb` the three Whisper models returned no text for the 219 clips longer than 30 s (`whisper_empty_segments.json`). This notebook decodes them again: `asr_chunks.split()` cuts each clip into pieces of at most 28 s at its quietest points, and each piece is decoded as in `NB_v10_asr.ipynb` (`language=german`, bf16). The notebook writes `whisper_long_redecode/<system>.json` after every clip and skips clips that are already done, so it can be run again after a crash. It takes about 15 minutes.

Then continue with step 3 of `README.md`: `python whisper_long_redecode.py --merge`.

In [ ]:
import os, sys, json, time
from pathlib import Path
import torch, soundfile as sf
sys.path.insert(0, os.getcwd())
import asr_chunks
EMPTY = json.loads(Path("whisper_empty_segments.json").read_text())
CLIPS = Path("goldset/clips_v10")
miss = [k for k in EMPTY if not (CLIPS / f"{k}.mp3").exists()]
assert not miss, f"{len(miss)} clips missing, e.g. {miss[:3]} - unzip clips_v10_gold.zip and clips_v10_rest.zip"
assert torch.cuda.is_available(), "no GPU"
print(len(EMPTY), "clips ready - environment OK")

In [ ]:
from transformers import AutoModelForSpeechSeq2Seq, AutoProcessor, pipeline
MODELS = {"whisper_large_v3": "openai/whisper-large-v3",
          "whisper_turbo": "openai/whisper-large-v3-turbo",
          "whisper_turbo_de": "primeline/whisper-large-v3-turbo-german"}
for SYS, MID in MODELS.items():
    kw = {"dtype": torch.bfloat16}
    try:
        m = AutoModelForSpeechSeq2Seq.from_pretrained(MID, low_cpu_mem_usage=True, **kw).to("cuda")
    except TypeError:
        kw = {"torch_dtype": torch.bfloat16}
        m = AutoModelForSpeechSeq2Seq.from_pretrained(MID, low_cpu_mem_usage=True, **kw).to("cuda")
    pr = AutoProcessor.from_pretrained(MID)
    pipe = pipeline("automatic-speech-recognition", model=m, tokenizer=pr.tokenizer,
                    feature_extractor=pr.feature_extractor, device="cuda", **kw)
    OUT = Path("whisper_long_redecode"); OUT.mkdir(exist_ok=True)
    P = OUT / f"{SYS}.json"
    done = json.loads(P.read_text()) if P.exists() else {}
    t0 = time.time()
    for i, k in enumerate([k for k in EMPTY if k not in done], 1):
        a, sr = sf.read(str(CLIPS / f"{k}.mp3"), dtype="float32")
        if a.ndim > 1: a = a.mean(axis=1)
        pieces = asr_chunks.split(a, sr)
        txt = asr_chunks.join([pipe({"raw": p, "sampling_rate": sr}, generate_kwargs={"language": "german"})["text"] for p in pieces])
        done[k] = {"text": txt, "pieces": len(pieces), "seconds": round(len(a) / sr, 1)}
        P.write_text(json.dumps(done, ensure_ascii=False, indent=1))
        if i % 20 == 0: print(i, "done", f"{time.time()-t0:.0f}s", flush=True)
    print("finished:", len(done), "clips; empty results:", sum(not v["text"].strip() for v in done.values()))
    print(SYS, "saved")
    del pipe, m; torch.cuda.empty_cache()
print("ALL DONE - download the folder whisper_long_redecode (3 json files)")